# Reversal features vs IMDb rating

Replicates the Study 1 (movies) analysis of Knight et al. (2024) for every scorer/dimension:
summary stats, Pearson correlations, Table 1 regression, scatter plots, and Fig. 1 trajectories.

Run `plot-shift extract-reversals` first. The detector settings below must match the ones used for extraction.

In [ ]:
from plot_shift.data.db import PlotShiftSqliteDB
from plot_shift.evaluation import reversal_report as rr
from plot_shift.reversals import create_reversal_detector

DB_PATH = "../data/proceeded/data.db"
detector = create_reversal_detector("ruptures", window_size=5, ruptures_penalty=1.0)

db = PlotShiftSqliteDB(DB_PATH).__enter__()
df = rr.load_features(db)
df.head()

## Summary statistics

Knight (movies): reversals M=14.46, SD=2.78; magnitude M=0.06, SD=0.02; rating M=6.44, SD=1.02.

In [ ]:
rr.summary_stats(df).round(3)

## Pearson correlation with IMDb rating

In [ ]:
rr.pearson_table(df).pivot(index=rr.GROUP, columns="feature", values=["r", "p"]).round(3)

## Table 1: rating ~ z(n_reversals) + z(mean_intensity)

Knight (movies): n_reversals 0.166\*\*\*, magnitude 0.039\*, adjusted R² 0.021.

In [ ]:
rr.regression_table(df).round(4)

## Features vs rating

In [ ]:
for feature in rr.FEATURES:
    rr.plot_feature_vs_rating(df, feature)

## Per-movie comparison (sorted by rating)

In [ ]:
rr.movie_table(df, "n_reversals")

In [ ]:
rr.movie_table(df, "mean_intensity").round(3)

## Fig. 1: trajectories of the highest- and lowest-rated movies

Purple = up trend, blue = down trend, red = reversal extrema. Change `movie_ids` to inspect any movie.

In [ ]:
movies = df.drop_duplicates("imdb_id_int").set_index("imdb_id_int").sort_values(rr.RATING)
movie_ids = [movies.index[-1], movies.index[0]]

for imdb_id_int in movie_ids:
    m = movies.loc[imdb_id_int]
    rr.plot_movie(db, imdb_id_int, detector, title=f"{m.primary_title} ({m.start_year}), IMDb {m.average_rating}")